In [1]:
!pip uninstall boto3 botocore s3transfer -y

In [2]:
pip install --no-cache-dir "pyiceberg[s3fs,duckdb,pandas,pyiceberg-core]" boto3 botocore

INFO: pip is looking at multiple versions of aiobotocore to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of aiobotocore to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of s3fs to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of s3fs to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/

In [3]:
from pyiceberg.catalog.sql import SqlCatalog

In [4]:
import pyarrow as pa
import pandas as pd
import os
from datetime import datetime, timezone

In [5]:
# Criar o diretório para os metadados do catálogo
os.makedirs("../data/iceberg_catalog", exist_ok=True)

# Configurar o catálogo SQLite local
# Em produção, o catálogo seria um banco de dados externo (PostgreSQL, Hive Metastore)
# ou um serviço gerenciado (AWS Glue, Nessie, Polaris)
### CONEXÃO MAQUINA HOST
# catalog = SqlCatalog(
#     "curso_lakehouse",
#     **{
#         "uri": "sqlite:///../data/iceberg_catalog/catalog.db",
#         "warehouse": "s3://silver",  # As tabelas Iceberg serão armazenadas no bucket silver
#         "s3.endpoint": "http://localhost:9000",
#        "s3.access-key-id": "minioadmin",
#        "s3.secret-access-key": "minioadmin123",
#        "s3.path-style-access": "true",
#    },
#)

### CONEXÃO ENTRE CONTAINERS DOCKERs
catalog = SqlCatalog(
    "curso_lakehouse",
    **{
        "uri": "sqlite:///../data/iceberg_catalog/catalog.db",
        "warehouse": "s3://silver",  # As tabelas Iceberg serão armazenadas no bucket silver
        "s3.endpoint": "http://host.docker.internal:9000",
        "s3.access-key-id": "minioadmin",
        "s3.secret-access-key": "minioadmin123",
        "s3.path-style-access": "true",
    },
)


# Criar o namespace (equivalente a um schema/database)
try:
    catalog.create_namespace("bronze_processado")
    print("✅ Namespace 'bronze_processado' criado!")
except Exception:
    print("ℹ️  Namespace já existe, continuando...")
 
print(f"\nNamespaces disponíveis: {catalog.list_namespaces()}")

ℹ️  Namespace já existe, continuando...

Namespaces disponíveis: [('bronze_processado',)]


In [6]:
pip install "pyiceberg[pyiceberg-core]"

Note: you may need to restart the kernel to use updated packages.


In [7]:
# Célula 2 — Definição do schema e criação da tabela Iceberg

from pyiceberg.schema import Schema
from pyiceberg.types import (
    NestedField, IntegerType, StringType, DoubleType,
    TimestampType, BooleanType, LongType
)
from pyiceberg.partitioning import PartitionSpec, PartitionField
from pyiceberg.transforms import MonthTransform, IdentityTransform

# Definir o schema da tabela (fortemente tipado)
schema_pedidos = Schema(
    NestedField(field_id=1,  name="id_pedido",      field_type=LongType(),      required=True),
    NestedField(field_id=2,  name="id_cliente",     field_type=LongType(),      required=True),
    NestedField(field_id=3,  name="id_produto",     field_type=LongType(),      required=True),
    NestedField(field_id=4,  name="categoria",      field_type=StringType(),    required=False),
    NestedField(field_id=5,  name="valor_unitario", field_type=DoubleType(),    required=False),
    NestedField(field_id=6,  name="quantidade",     field_type=IntegerType(),   required=False),
    NestedField(field_id=7,  name="valor_total",    field_type=DoubleType(),    required=False),
    NestedField(field_id=8,  name="status_pedido",  field_type=StringType(),    required=False),
    NestedField(field_id=9,  name="regiao",         field_type=StringType(),    required=False),
    NestedField(field_id=10, name="data_pedido",    field_type=TimestampType(), required=False),
    NestedField(field_id=11, name="avaliacao_cliente", field_type=IntegerType(), required=False),
    # Metadados de rastreabilidade
    NestedField(field_id=12, name="_fonte",         field_type=StringType(),    required=False),
    NestedField(field_id=13, name="_ingerido_em",   field_type=StringType(),    required=False),
)

# Definir a estratégia de particionamento
# MonthTransform particiona por mês da coluna data_pedido
spec_particao = PartitionSpec(
    PartitionField(
        source_id=10,           # Campo data_pedido (field_id=10)
        field_id=1000,
        transform=MonthTransform(),
        name="data_pedido_mes",
    )
)

# Criar a tabela no catálogo
NOME_TABELA = "bronze_processado.pedidos_ecommerce"

try:
    tabela = catalog.create_table(
        identifier=NOME_TABELA,
        schema=schema_pedidos,
        partition_spec=spec_particao,
        properties={
            "write.format.default": "parquet",
            "write.parquet.compression-codec": "snappy",
            "write.metadata.compression-codec": "gzip",
        },
    )
    print(f"✅ Tabela Iceberg criada: {NOME_TABELA}")
except Exception:
    tabela = catalog.load_table(NOME_TABELA)
    print(f"ℹ️  Tabela já existe, carregada: {NOME_TABELA}")

print(f"\nSchema da tabela:")
print(tabela.schema())
print(f"\nEspecificação de particionamento:")
print(tabela.spec())

ℹ️  Tabela já existe, carregada: bronze_processado.pedidos_ecommerce

Schema da tabela:
table {
  1: id_pedido: required long
  2: id_cliente: required long
  3: id_produto: required long
  4: categoria: optional string
  5: valor_unitario: optional double
  6: quantidade: optional int
  7: valor_total: optional double
  8: status_pedido: optional string
  9: regiao: optional string
  10: data_pedido: optional timestamp
  11: avaliacao_cliente: optional int
  12: _fonte: optional string
  13: _ingerido_em: optional string
}

Especificação de particionamento:
[
  1000: data_pedido_mes: month(10)
]


In [8]:
import boto3
import io
import pyarrow.parquet as pq

# Ler os dados do MinIO (camada Bronze)
s3_client = boto3.client(
    "s3",
    endpoint_url="http://host.docker.internal:9000",
    aws_access_key_id="minioadmin",
    aws_secret_access_key="minioadmin123",
    region_name="us-east-1",
)

print("Lendo dados da camada Bronze...")
response = s3_client.list_objects_v2(
    Bucket="bronze", Prefix="ecommerce_sintetico/"
)
arquivos = [
    obj["Key"] for obj in response.get("Contents", [])
    if obj["Key"].endswith(".parquet")
]

# Ler o primeiro arquivo Parquet (amostra de 50.000 registros para a demo)
obj = s3_client.get_object(Bucket="bronze", Key=arquivos[0])
df_bronze = pd.read_parquet(io.BytesIO(obj["Body"].read()))

# Selecionar apenas os primeiros 50.000 registros para a demonstração
df_amostra = df_bronze.head(50_000).copy()

# Ajustar tipos para compatibilidade com o schema Iceberg
df_amostra["id_pedido"] = df_amostra["id_pedido"].astype("int64")
df_amostra["id_cliente"] = df_amostra["id_cliente"].astype("int64")
df_amostra["id_produto"] = df_amostra["id_produto"].astype("int64")
df_amostra["avaliacao_cliente"] = pd.to_numeric(
    df_amostra["avaliacao_cliente"], errors="coerce"
).astype("Int32")

# Converter para PyArrow Table (formato nativo do Iceberg)
tabela_arrow = pa.Table.from_pandas(
    df_amostra[[
        "id_pedido", "id_cliente", "id_produto", "categoria",
        "valor_unitario", "quantidade", "valor_total", "status_pedido",
        "regiao", "data_pedido", "avaliacao_cliente", "_fonte", "_ingerido_em"
    ]],
    schema=tabela.schema().as_arrow(),
    preserve_index=False,
)

# Inserir os dados na tabela Iceberg (cria o Snapshot 1)
print(f"Inserindo {len(df_amostra):,} registros na tabela Iceberg...")
import time
inicio = time.time()

tabela.append(tabela_arrow)

duracao = time.time() - inicio
print(f"✅ Snapshot 1 criado em {duracao:.2f}s")
print(f"   Registros inseridos: {len(df_amostra):,}")

Lendo dados da camada Bronze...
Inserindo 50,000 registros na tabela Iceberg...
✅ Snapshot 1 criado em 0.23s
   Registros inseridos: 50,000


In [9]:
import boto3
import io
import pyarrow.parquet as pq

# Ler os dados do MinIO (camada Bronze)
s3_client = boto3.client(
    "s3",
    endpoint_url="http://host.docker.internal:9000",
    aws_access_key_id="minioadmin",
    aws_secret_access_key="minioadmin123",
    region_name="us-east-1",
)

print("Lendo dados da camada Bronze...")
response = s3_client.list_objects_v2(
    Bucket="bronze", Prefix="ecommerce_sintetico/"
)
arquivos = [
    obj["Key"] for obj in response.get("Contents", [])
    if obj["Key"].endswith(".parquet")
]

# Ler o primeiro arquivo Parquet (amostra de 50.000 registros para a demo)
obj = s3_client.get_object(Bucket="bronze", Key=arquivos[0])
df_bronze = pd.read_parquet(io.BytesIO(obj["Body"].read()))

# Selecionar apenas os primeiros 50.000 registros para a demonstração
df_amostra = df_bronze.head(50_000).copy()

# Ajustar tipos para compatibilidade com o schema Iceberg
df_amostra["id_pedido"] = df_amostra["id_pedido"].astype("int64")
df_amostra["id_cliente"] = df_amostra["id_cliente"].astype("int64")
df_amostra["id_produto"] = df_amostra["id_produto"].astype("int64")
df_amostra["avaliacao_cliente"] = pd.to_numeric(
    df_amostra["avaliacao_cliente"], errors="coerce"
).astype("Int32")

# Converter para PyArrow Table (formato nativo do Iceberg)
tabela_arrow = pa.Table.from_pandas(
    df_amostra[[
        "id_pedido", "id_cliente", "id_produto", "categoria",
        "valor_unitario", "quantidade", "valor_total", "status_pedido",
        "regiao", "data_pedido", "avaliacao_cliente", "_fonte", "_ingerido_em"
    ]],
    schema=tabela.schema().as_arrow(),
    preserve_index=False,
)

# Inserir os dados na tabela Iceberg (cria o Snapshot 1)
print(f"Inserindo {len(df_amostra):,} registros na tabela Iceberg...")

tabela.append(tabela_arrow)



Lendo dados da camada Bronze...
Inserindo 50,000 registros na tabela Iceberg...


In [10]:
# Célula 4 — Inspecionar o histórico de snapshots

print("=== Histórico de Snapshots da Tabela Iceberg ===\n")

for snapshot in tabela.snapshots():
    print(f"  Snapshot ID:    {snapshot.snapshot_id}")
    print(f"  Timestamp:      {datetime.fromtimestamp(snapshot.timestamp_ms / 1000, tz=timezone.utc)}")
    print(f"  Operação:       {snapshot.summary.get('operation', 'N/A')}")
    print(f"  Arquivos adicionados: {snapshot.summary.get('added-data-files', '0')}")
    print(f"  Registros adicionados: {snapshot.summary.get('added-records', '0')}")
    print()

=== Histórico de Snapshots da Tabela Iceberg ===

  Snapshot ID:    4299226873463529944
  Timestamp:      2026-06-25 22:56:52.602000+00:00
  Operação:       Operation.APPEND
  Arquivos adicionados: 1
  Registros adicionados: 50000

  Snapshot ID:    5565359227732382984
  Timestamp:      2026-06-25 22:56:54.665000+00:00
  Operação:       Operation.APPEND
  Arquivos adicionados: 1
  Registros adicionados: 50000



In [11]:
# Célula 5 — Inserir mais dados para criar o Snapshot 2

# Simular uma segunda carga de dados (próximo lote)
df_lote2 = df_bronze.iloc[50_000:100_000].copy()
df_lote2["id_pedido"] = df_lote2["id_pedido"].astype("int64")
df_lote2["id_cliente"] = df_lote2["id_cliente"].astype("int64")
df_lote2["id_produto"] = df_lote2["id_produto"].astype("int64")
df_lote2["avaliacao_cliente"] = pd.to_numeric(
    df_lote2["avaliacao_cliente"], errors="coerce"
).astype("Int32")

tabela_arrow_lote2 = pa.Table.from_pandas(
    df_lote2[[
        "id_pedido", "id_cliente", "id_produto", "categoria",
        "valor_unitario", "quantidade", "valor_total", "status_pedido",
        "regiao", "data_pedido", "avaliacao_cliente", "_fonte", "_ingerido_em"
    ]],
    schema=tabela.schema().as_arrow(),
    preserve_index=False,
)

tabela.append(tabela_arrow_lote2)
print(f"✅ Snapshot 2 criado: +{len(df_lote2):,} registros inseridos")

# Recarregar a tabela para ver o histórico atualizado
tabela = catalog.load_table(NOME_TABELA)
snapshots = list(tabela.snapshots())
print(f"\nTotal de snapshots: {len(snapshots)}")
for snap in snapshots:
    ts = datetime.fromtimestamp(snap.timestamp_ms / 1000, tz=timezone.utc)
    print(f"  Snapshot {snap.snapshot_id}: {ts.strftime('%H:%M:%S')} — {snap.summary.get('added-records', '0')} registros adicionados")

✅ Snapshot 2 criado: +50,000 registros inseridos

Total de snapshots: 3
  Snapshot 4299226873463529944: 22:56:52 — 50000 registros adicionados
  Snapshot 5565359227732382984: 22:56:54 — 50000 registros adicionados
  Snapshot 3657116537687466752: 22:56:54 — 50000 registros adicionados


In [12]:
# Célula 6 — Time Travel: consultar o estado da tabela em um snapshot anterior

# Obter o ID do primeiro snapshot
snapshot_inicial = snapshots[-1]  # O mais antigo é o último na lista
snapshot_id_v1 = snapshot_inicial.snapshot_id

print(f"=== Time Travel — Consultando Snapshot {snapshot_id_v1} ===\n")

# Ler a tabela no estado do snapshot inicial (apenas os primeiros 50.000 registros)
scan_v1 = tabela.scan(snapshot_id=snapshot_id_v1)
df_v1 = scan_v1.to_pandas()

# Ler a tabela no estado atual (todos os 100.000 registros)
scan_atual = tabela.scan()
df_atual = scan_atual.to_pandas()

print(f"  Snapshot inicial (v1): {len(df_v1):,} registros")
print(f"  Estado atual:          {len(df_atual):,} registros")
print(f"  Diferença:             +{len(df_atual) - len(df_v1):,} registros adicionados")
print()
print("💡 O Time Travel permite auditar o estado dos dados em qualquer")
print("   ponto do passado — essencial para debugging de pipelines e")
print("   reprodutibilidade de experimentos de Machine Learning.")

=== Time Travel — Consultando Snapshot 3657116537687466752 ===

  Snapshot inicial (v1): 150,000 registros
  Estado atual:          150,000 registros
  Diferença:             +0 registros adicionados

💡 O Time Travel permite auditar o estado dos dados em qualquer
   ponto do passado — essencial para debugging de pipelines e
   reprodutibilidade de experimentos de Machine Learning.


In [13]:
# Célula 7 — Evolução de Schema: adicionar uma nova coluna sem reescrever os dados

from pyiceberg.schema import Schema
from pyiceberg.types import NestedField, BooleanType

print("=== Evolução de Schema ===\n")
print(f"Schema ANTES da evolução:")
print(tabela.schema())

# Adicionar uma nova coluna à tabela Iceberg
# Em Parquet puro, isso exigiria reescrever TODOS os arquivos
# No Iceberg, é uma operação de metadados — instantânea!
with tabela.update_schema() as update:
    update.add_column(
        path="pedido_internacional",
        field_type=BooleanType(),
        doc="Indica se o pedido foi realizado por cliente de outro país",
    )

# Recarregar a tabela para ver o schema atualizado
tabela = catalog.load_table(NOME_TABELA)

print(f"\nSchema APÓS a evolução:")
print(tabela.schema())
print()
print("✅ Nova coluna 'pedido_internacional' adicionada!")
print("   Os arquivos Parquet existentes NÃO foram reescritos.")
print("   O Iceberg retorna NULL para a nova coluna nos registros antigos.")

=== Evolução de Schema ===

Schema ANTES da evolução:
table {
  1: id_pedido: required long
  2: id_cliente: required long
  3: id_produto: required long
  4: categoria: optional string
  5: valor_unitario: optional double
  6: quantidade: optional int
  7: valor_total: optional double
  8: status_pedido: optional string
  9: regiao: optional string
  10: data_pedido: optional timestamp
  11: avaliacao_cliente: optional int
  12: _fonte: optional string
  13: _ingerido_em: optional string
}

Schema APÓS a evolução:
table {
  1: id_pedido: required long
  2: id_cliente: required long
  3: id_produto: required long
  4: categoria: optional string
  5: valor_unitario: optional double
  6: quantidade: optional int
  7: valor_total: optional double
  8: status_pedido: optional string
  9: regiao: optional string
  10: data_pedido: optional timestamp
  11: avaliacao_cliente: optional int
  12: _fonte: optional string
  13: _ingerido_em: optional string
  14: pedido_internacional: optional bo

In [14]:
# Célula 8 — Consultar a tabela Iceberg com DuckDB
# O DuckDB tem suporte nativo ao Iceberg via extensão

import duckdb

con_duck = duckdb.connect()
con_duck.execute("INSTALL iceberg; LOAD iceberg;")
con_duck.execute("INSTALL httpfs; LOAD httpfs;")
con_duck.execute("""
    SET s3_endpoint = 'host.docker.internal:9000';
    SET s3_access_key_id = 'minioadmin';
    SET s3_secret_access_key = 'minioadmin123';
    SET s3_use_ssl = false;
    SET s3_url_style = 'path';
""")

# Obter o caminho do metadata.json da tabela Iceberg
# O PyIceberg armazena os metadados no bucket silver
metadata_location = tabela.metadata_location
print(f"Localização dos metadados Iceberg: {metadata_location}\n")

# Consultar a tabela Iceberg via DuckDB
resultado = con_duck.execute(f"""
    SELECT
        categoria,
        regiao,
        COUNT(*)                        AS total_pedidos,
        ROUND(SUM(valor_total), 2)      AS receita_total,
        ROUND(AVG(avaliacao_cliente), 2) AS satisfacao_media
    FROM iceberg_scan('{metadata_location}')
    WHERE status_pedido = 'concluido'
    GROUP BY categoria, regiao
    ORDER BY receita_total DESC
    LIMIT 10
""").df()

print("Consulta DuckDB sobre tabela Iceberg:")
print(resultado.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Localização dos metadados Iceberg: s3://silver/bronze_processado/pedidos_ecommerce/metadata/00004-241d5616-c169-43fc-8036-c7444e164ef3.metadata.json

Consulta DuckDB sobre tabela Iceberg:
  categoria       regiao  total_pedidos  receita_total  satisfacao_media
  Alimentos     Nordeste           4568    25658082.89              3.89
   Esportes          Sul           4533    25585141.96              3.93
     Roupas          Sul           4512    25541982.32              3.91
Eletrônicos          Sul           4636    25491388.00              3.94
   Esportes Centro-Oeste           4599    25319624.00              3.90
     Roupas Centro-Oeste           4588    25318004.67              3.93
Eletrônicos        Norte           4565    25233280.04              3.90
     Livros Centro-Oeste           4493    25085748.94              3.88
     Livros        Norte           4499    25057459.68              3.91
     Roupas      Sudeste           4515    24786036.22              3.90


In [15]:
# Célula 9 — Inspecionar a estrutura de arquivos do Iceberg no MinIO

import boto3

s3 = boto3.client(
    "s3",
    endpoint_url="http://host.docker.internal:9000",
    aws_access_key_id="minioadmin",
    aws_secret_access_key="minioadmin123",
    region_name="us-east-1",
)

print("=== Estrutura de Arquivos do Iceberg no MinIO (bucket: silver) ===\n")

response = s3.list_objects_v2(Bucket="silver", Prefix="bronze_processado/")
objetos = response.get("Contents", [])

# Agrupar por tipo de arquivo
metadados = [o for o in objetos if "/metadata/" in o["Key"]]
dados = [o for o in objetos if "/data/" in o["Key"]]

print(f"📁 Arquivos de METADADOS ({len(metadados)} arquivos):")
for obj in metadados:
    tipo = "snapshot" if "snap-" in obj["Key"] else \
           "schema" if "schema" in obj["Key"] else \
           "manifest" if "manifest" in obj["Key"] else "metadata"
    print(f"   [{tipo:10s}] {obj['Key'].split('/')[-1]}  ({obj['Size'] / 1024:.1f} KB)")

print(f"\n📁 Arquivos de DADOS ({len(dados)} arquivos Parquet):")
for obj in dados:
    print(f"   [parquet   ] {obj['Key'].split('/')[-1]}  ({obj['Size'] / 1024:.1f} KB)")

print(f"""
💡 Estrutura do Iceberg:
   metadata/  → Histórico de snapshots, schemas e manifests (metadados)
   data/      → Arquivos Parquet com os dados reais

   O Iceberg NUNCA modifica arquivos de dados existentes.
   Cada operação de escrita cria NOVOS arquivos e um NOVO snapshot,
   garantindo leituras consistentes mesmo durante escritas concorrentes.
""")

=== Estrutura de Arquivos do Iceberg no MinIO (bucket: silver) ===

📁 Arquivos de METADADOS (15 arquivos):
   [metadata  ] 00000-18042cbc-d4c4-418e-9781-705c2caddaaf.metadata.json  (1.6 KB)
   [metadata  ] 00000-6331a69c-e3bd-4013-951b-967f5c1033c7.metadata.json  (1.6 KB)
   [metadata  ] 00000-a9ee2f6f-2568-4e52-9d2d-c99962e3f24d.metadata.json  (1.6 KB)
   [metadata  ] 00000-b831400b-18af-4e4b-9487-4a9c22c13e47.metadata.json  (1.6 KB)
   [metadata  ] 00000-e49e95f5-c5b4-473b-a41d-48348c279cfe.metadata.json  (1.6 KB)
   [metadata  ] 00001-caf9682d-a14a-431d-acfc-d089ca273990.metadata.json  (2.4 KB)
   [metadata  ] 00002-ed20f97e-8bfd-4e94-8f56-bb93eba2be73.metadata.json  (3.2 KB)
   [metadata  ] 00003-8703ea01-5620-4032-b051-eace6ec56e65.metadata.json  (4.0 KB)
   [metadata  ] 00004-241d5616-c169-43fc-8036-c7444e164ef3.metadata.json  (5.1 KB)
   [metadata  ] 10f2efca-b582-4d10-9de1-b79f70ce26c4-m0.avro  (5.3 KB)
   [metadata  ] a292efcf-9de7-40c2-9a84-c82f27ae323e-m0.avro  (5.3 KB)
   [